# تشخیص لبه کنی / Canny Edge Detection

**دانشجو / Student:** مسیح معافی / Masih Moafi  
**تاریخ / Date:** ۱۴۰۳/۰۷/۲۸ (October 19, 2025)

---

## هدف / Objective

**هدف:** یادگیری الگوریتم تشخیص لبه کنی و تنظیم پارامترهای آن

**Objective:** Learn the Canny edge detection algorithm and parameter tuning

---

## محتوا / Contents

1. مقدمه‌ای بر الگوریتم کنی / Introduction to Canny Algorithm
2. مراحل الگوریتم کنی / Steps of Canny Algorithm
3. پیاده‌سازی ساده / Simple Implementation
4. تأثیر آستانه‌ها / Effect of Thresholds
5. مراحل میانی / Intermediate Steps
6. تنظیم پارامترها / Parameter Tuning
7. مقایسه با روش‌های دیگر / Comparison with Other Methods
8. کاربردهای عملی / Practical Applications

In [ ]:
# وارد کردن کتابخانه‌های مورد نیاز / Import required libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List
import os

# تنظیمات نمایش / Display settings
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. توابع کمکی / Helper Functions

In [ ]:
def display_images(images: list, titles: list, cmap='gray', rows=1):
    """
    نمایش چند تصویر در کنار هم
    """
    n = len(images)
    cols = (n + rows - 1) // rows
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
    
    for i, (img, title) in enumerate(zip(images, titles)):
        axes[i].imshow(img, cmap=cmap)
        axes[i].set_title(title, fontsize=14, fontweight='bold')
        axes[i].axis('off')
    
    for i in range(n, len(axes)):
        axes[i].remove()
    
    plt.tight_layout()
    plt.show()

print("✓ توابع کمکی آماده شدند")

## 2. ایجاد تصویر نمونه / Create Sample Image

In [ ]:
# ایجاد تصویر نمونه
# Create sample image
image = np.zeros((400, 600), dtype=np.uint8)

# اضافه کردن اشکال مختلف
# Add different shapes
cv2.rectangle(image, (50, 50), (200, 200), 255, -1)
cv2.rectangle(image, (250, 100), (400, 250), 200, -1)
cv2.circle(image, (150, 300), 60, 180, -1)
cv2.circle(image, (400, 320), 50, 220, -1)
cv2.ellipse(image, (500, 150), (60, 40), 45, 0, 360, 150, -1)

# اضافه کردن نویز خفیف
# Add slight noise
noise = np.random.normal(0, 5, image.shape)
noisy_image = np.clip(image.astype(np.float64) + noise, 0, 255).astype(np.uint8)

display_images([image, noisy_image], 
               ['تصویر اصلی / Original', 'تصویر نویزی / Noisy'])

print(f"اندازه تصویر / Image size: {image.shape}")

## 3. الگوریتم کنی - مقدمه / Canny Algorithm - Introduction

الگوریتم کنی یک روش چند مرحله‌ای برای تشخیص لبه است:

The Canny algorithm is a multi-stage edge detection method:

1. **محو گاوسی / Gaussian Blur**: کاهش نویز
2. **محاسبه گرادیان / Gradient Calculation**: یافتن قدرت و جهت لبه
3. **سرکوب غیرماکزیمم / Non-Maximum Suppression**: نازک کردن لبه‌ها
4. **آستانه‌گذاری هیسترزیس / Hysteresis Thresholding**: انتخاب لبه‌های قوی و ضعیف

## 4. پیاده‌سازی ساده / Simple Implementation

In [ ]:
# اعمال تشخیص لبه کنی
# Apply Canny edge detection
edges = cv2.Canny(noisy_image, threshold1=50, threshold2=150)

# نمایش نتیجه
# Display result
display_images([noisy_image, edges],
               ['تصویر اصلی\nOriginal', 'لبه‌های کنی\nCanny Edges'])

print("✓ تشخیص لبه کنی انجام شد")
print(f"تعداد پیکسل‌های لبه / Edge pixels: {np.sum(edges > 0)}")
print(f"درصد لبه / Edge percentage: {100 * np.sum(edges > 0) / edges.size:.2f}%")

## 5. تأثیر آستانه‌ها / Effect of Thresholds

دو آستانه در الگوریتم کنی:
- **threshold1 (آستانه پایین)**: لبه‌های ضعیف
- **threshold2 (آستانه بالا)**: لبه‌های قوی

Two thresholds in Canny algorithm:
- **threshold1 (low threshold)**: weak edges
- **threshold2 (high threshold)**: strong edges

In [ ]:
# آزمایش آستانه‌های مختلف
# Test different thresholds
threshold_pairs = [(30, 90), (50, 150), (100, 200), (150, 250)]
results = []
titles = ['تصویر اصلی\nOriginal']

for t1, t2 in threshold_pairs:
    edges = cv2.Canny(noisy_image, t1, t2)
    results.append(edges)
    edge_count = np.sum(edges > 0)
    titles.append(f'T1={t1}, T2={t2}\nEdges: {edge_count}')

all_images = [noisy_image] + results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (img, title) in enumerate(zip(all_images, titles)):
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(title, fontsize=12, fontweight='bold')
    axes[i].axis('off')

axes[-1].remove()

plt.tight_layout()
plt.show()

print("✓ تأثیر آستانه‌ها نمایش داده شد")

## 6. مراحل میانی الگوریتم / Intermediate Steps

In [ ]:
def canny_steps(image: np.ndarray, t1: int = 50, t2: int = 150) -> dict:
    """
    نمایش مراحل میانی الگوریتم کنی
    
    Args:
        image: تصویر ورودی
        t1: آستانه پایین
        t2: آستانه بالا
    
    Returns:
        دیکشنری حاوی مراحل مختلف
    """
    # مرحله 1: محو گاوسی
    # Step 1: Gaussian blur
    blurred = cv2.GaussianBlur(image, (5, 5), 1.4)
    
    # مرحله 2: محاسبه گرادیان
    # Step 2: Calculate gradients
    gx = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(gx**2 + gy**2)
    direction = np.arctan2(gy, gx)
    
    # نرمال‌سازی قدرت گرادیان
    # Normalize magnitude
    magnitude_norm = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    # مرحله 3 و 4: سرکوب غیرماکزیمم و هیسترزیس (توسط cv2.Canny)
    # Steps 3 & 4: Non-maximum suppression and hysteresis (by cv2.Canny)
    edges = cv2.Canny(image, t1, t2)
    
    return {
        'original': image,
        'blurred': blurred,
        'magnitude': magnitude_norm,
        'edges': edges
    }


# نمایش مراحل
# Display steps
steps = canny_steps(noisy_image, 50, 150)

images = [steps['original'], steps['blurred'], steps['magnitude'], steps['edges']]
titles = ['۱. تصویر اصلی\n1. Original',
          '۲. محو گاوسی\n2. Gaussian Blur',
          '۳. قدرت گرادیان\n3. Gradient Magnitude',
          '۴. لبه‌های نهایی\n4. Final Edges']

display_images(images, titles, rows=2)

print("✓ مراحل میانی نمایش داده شد")

## 7. تنظیم خودکار آستانه‌ها / Automatic Threshold Tuning

یک روش رایج: استفاده از میانه قدرت گرادیان

A common method: using the median of gradient magnitude

In [ ]:
def auto_canny(image: np.ndarray, sigma: float = 0.33) -> Tuple[np.ndarray, int, int]:
    """
    تشخیص لبه کنی با آستانه‌های خودکار
    
    Args:
        image: تصویر ورودی
        sigma: ضریب برای محاسبه آستانه‌ها
    
    Returns:
        لبه‌ها، آستانه پایین، آستانه بالا
    """
    # محاسبه میانه
    # Calculate median
    median = np.median(image)
    
    # محاسبه آستانه‌ها
    # Calculate thresholds
    lower = int(max(0, (1.0 - sigma) * median))
    upper = int(min(255, (1.0 + sigma) * median))
    
    # اعمال کنی
    # Apply Canny
    edges = cv2.Canny(image, lower, upper)
    
    return edges, lower, upper


# تست روی تصاویر مختلف
# Test on different images
test_images = [noisy_image, image]
results = []

for img in test_images:
    # دستی
    # Manual
    manual_edges = cv2.Canny(img, 50, 150)
    
    # خودکار
    # Automatic
    auto_edges, t1, t2 = auto_canny(img)
    
    results.append((img, manual_edges, auto_edges, t1, t2))

# نمایش نتایج
# Display results
for i, (img, manual, auto, t1, t2) in enumerate(results):
    images = [img, manual, auto]
    titles = [f'تصویر {i+1}\nImage {i+1}',
              'دستی (50, 150)\nManual (50, 150)',
              f'خودکار ({t1}, {t2})\nAuto ({t1}, {t2})']
    display_images(images, titles)

print("✓ تنظیم خودکار آستانه‌ها انجام شد")

## 8. مقایسه با روش‌های دیگر / Comparison with Other Methods

In [ ]:
# آماده‌سازی تصویر
# Prepare image
blurred = cv2.GaussianBlur(noisy_image, (5, 5), 0)

# کنی
# Canny
canny = cv2.Canny(blurred, 50, 150)

# سوبل
# Sobel
sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
sobel = np.sqrt(sobel_x**2 + sobel_y**2)
sobel = cv2.normalize(sobel, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
_, sobel_binary = cv2.threshold(sobel, 50, 255, cv2.THRESH_BINARY)

# لاپلاسین
# Laplacian
laplacian = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
laplacian = np.abs(laplacian)
laplacian = cv2.normalize(laplacian, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
_, laplacian_binary = cv2.threshold(laplacian, 50, 255, cv2.THRESH_BINARY)

# نمایش مقایسه
# Display comparison
images = [noisy_image, canny, sobel_binary, laplacian_binary]
titles = ['تصویر اصلی\nOriginal',
          'کنی\nCanny',
          'سوبل\nSobel',
          'لاپلاسین\nLaplacian']

display_images(images, titles, rows=2)

# محاسبه آمار
# Calculate statistics
print("\nآمار لبه‌ها / Edge Statistics:")
print(f"کنی / Canny: {np.sum(canny > 0)} پیکسل")
print(f"سوبل / Sobel: {np.sum(sobel_binary > 0)} پیکسل")
print(f"لاپلاسین / Laplacian: {np.sum(laplacian_binary > 0)} پیکسل")

## 9. تأثیر محو گاوسی / Effect of Gaussian Blur

In [ ]:
# ایجاد تصویر نویزی
# Create noisy image
noise = np.random.normal(0, 15, image.shape)
very_noisy = np.clip(image.astype(np.float64) + noise, 0, 255).astype(np.uint8)

# کنی بدون محو
# Canny without blur
canny_no_blur = cv2.Canny(very_noisy, 50, 150)

# کنی با محوهای مختلف
# Canny with different blurs
blur_sizes = [(3, 3), (5, 5), (7, 7)]
results = []

for size in blur_sizes:
    blurred = cv2.GaussianBlur(very_noisy, size, 0)
    edges = cv2.Canny(blurred, 50, 150)
    results.append(edges)

# نمایش نتایج
# Display results
images = [very_noisy, canny_no_blur] + results
titles = ['تصویر نویزی\nNoisy Image',
          'بدون محو\nNo Blur',
          'محو 3x3\nBlur 3x3',
          'محو 5x5\nBlur 5x5',
          'محو 7x7\nBlur 7x7']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (img, title) in enumerate(zip(images, titles)):
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(title, fontsize=12, fontweight='bold')
    axes[i].axis('off')

axes[-1].remove()

plt.tight_layout()
plt.show()

print("✓ تأثیر محو گاوسی نمایش داده شد")

## 10. کاربردهای عملی / Practical Applications

In [ ]:
# تست روی تصاویر واقعی
# Test on real images
test_paths = [
    '../chapter0_opencv_tutorial/New_Zealand_Lake.jpg',
    '../chapter0_opencv_tutorial/coca-cola-logo.png',
    '../chapter2_practical/how-many-horses.webp'
]

for path in test_paths:
    if os.path.exists(path):
        # خواندن تصویر
        # Read image
        img = cv2.imread(path)
        
        if img is not None:
            # تبدیل به خاکستری
            # Convert to grayscale
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            # تغییر اندازه در صورت نیاز
            # Resize if needed
            if gray.shape[0] > 500 or gray.shape[1] > 500:
                scale = 500 / max(gray.shape)
                gray = cv2.resize(gray, None, fx=scale, fy=scale)
            
            # تشخیص لبه با آستانه‌های مختلف
            # Edge detection with different thresholds
            edges1 = cv2.Canny(gray, 30, 90)
            edges2 = cv2.Canny(gray, 50, 150)
            edges3 = cv2.Canny(gray, 100, 200)
            
            # تشخیص لبه خودکار
            # Automatic edge detection
            auto_edges, t1, t2 = auto_canny(gray)
            
            # نمایش نتایج
            # Display results
            images = [gray, edges1, edges2, edges3, auto_edges]
            titles = ['تصویر اصلی\nOriginal',
                      '(30, 90)',
                      '(50, 150)',
                      '(100, 200)',
                      f'خودکار\nAuto ({t1}, {t2})']
            
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            axes = axes.flatten()
            
            for i, (img_show, title) in enumerate(zip(images, titles)):
                axes[i].imshow(img_show, cmap='gray')
                axes[i].set_title(title, fontsize=12, fontweight='bold')
                axes[i].axis('off')
            
            axes[-1].remove()
            
            plt.suptitle(f'تصویر: {os.path.basename(path)}', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()
            
            print(f"✓ تست روی تصویر واقعی انجام شد: {os.path.basename(path)}")
            break
else:
    print("⚠ تصویر تست یافت نشد")

## 11. تمرین‌ها / Exercises

### تمرین 1 / Exercise 1
یک تابع بنویسید که بهترین آستانه‌های کنی را برای یک تصویر پیدا کند.

Write a function that finds the best Canny thresholds for an image.

### تمرین 2 / Exercise 2
تصویری با نویز زیاد ایجاد کنید و ببینید چه اندازه محو گاوسی بهترین نتیجه را می‌دهد.

Create an image with high noise and see what Gaussian blur size gives the best result.

### تمرین 3 / Exercise 3
الگوریتم کنی را با سوبل و لاپلاسین روی چند تصویر مختلف مقایسه کنید.

Compare Canny algorithm with Sobel and Laplacian on several different images.

### تمرین 4 / Exercise 4
یک برنامه تعاملی بنویسید که کاربر بتواند آستانه‌های کنی را به‌صورت زنده تنظیم کند.

Write an interactive program where the user can adjust Canny thresholds in real-time.